In [3]:
# =========================================
# 0. INSTALAR BIBLIOTECA
# =========================================

!pip install geobr

# =========================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# =========================================

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import geobr
import os

# =========================================
# 2. CRIAR PASTAS
# =========================================

os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs/visualizacoes', exist_ok=True)

# =========================================
# 3. CARREGAR CSV DA PRF
# =========================================

df = pd.read_csv(
    'acidentes2023.csv',
    sep=';',
    encoding='latin1',
    on_bad_lines='skip',
    engine='python'
)

print(df.head())

# =========================================
# 4. FILTRAR RIO GRANDE DO SUL
# =========================================

rs = df[df['uf'] == 'RS'].copy()

print(rs.head())

# =========================================
# 5. CORRIGIR LATITUDE E LONGITUDE
# =========================================

rs['longitude'] = (
    rs['longitude']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

rs['latitude'] = (
    rs['latitude']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

print(rs[['longitude', 'latitude']].head())

# =========================================
# 6. CRIAR GEODATAFRAME
# =========================================

geometry = [
    Point(xy)
    for xy in zip(rs['longitude'], rs['latitude'])
]

gdf = gpd.GeoDataFrame(
    rs,
    geometry=geometry,
    crs='EPSG:4326'
)

print(gdf.head())

# =========================================
# 7. REPROJETAR PARA EPSG:31982
# =========================================

gdf = gdf.to_crs(epsg=31982)

print(gdf.crs)

# =========================================
# 8. CARREGAR MUNICÍPIOS DO RS
# =========================================

municipios = geobr.read_municipality(
    code_muni='RS',
    year=2020
)

municipios = municipios.to_crs(epsg=31982)

print(municipios.head())

# =========================================
# 9. JOIN ESPACIAL
# =========================================

joined = gpd.sjoin(
    gdf,
    municipios,
    how='left',
    predicate='within'
)

print(joined.head())

# =========================================
# 10. EXPORTAR GEOJSONS
# =========================================

joined.to_file(
    'acidentes-rs.geojson',
    driver='GeoJSON'
)

municipios.to_file(
    'municipios-rs.geojson',
    driver='GeoJSON'
)

print('GeoJSONs exportados com sucesso!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.0/338.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.9 MB/s eta 0:00:00
  Attempting uninstall: shapely
    Found existing installation: shapely 2.1.2
    Uninstalling shapely-2.1.2:
      Successfully uninstalled shapely-2.1.2
  Attempting uninstall: lxml
    Found existing installation: lxml 6.1.1
    Uninstalling lxml-6.1.1:
      Successfully uninstalled lxml-6.1.1
  Attempting uninstall: geopandas
    Found existing installation: geopandas 1.1.3
    Uninstalling geopandas-1.1.3:
      Successfully uninstalled geopandas-1.1.3


       id      pesid data_inversa dia_semana   horario  uf     br     km  \
0  496506  1082142.0   2023-01-01    domingo  00:15:00  MG  116.0    587   
1  496506  1082142.0   2023-01-01    domingo  00:15:00  MG  116.0    587   
2  496506  1082142.0   2023-01-01    domingo  00:15:00  MG  116.0    587   
3  496506  1082142.0   2023-01-01    domingo  00:15:00  MG  116.0    587   
4  496507  1082138.0   2023-01-01    domingo  00:20:00  MG  381.0  686,5   

  municipio causa_principal  ...       sexo  ilesos feridos_leves  \
0  MANHUACU             Não  ...  Masculino     0.0           1.0   
1  MANHUACU             Sim  ...  Masculino     0.0           1.0   
2  MANHUACU             Não  ...  Masculino     0.0           1.0   
3  MANHUACU             Sim  ...  Masculino     0.0           1.0   
4    LAVRAS             Sim  ...  Masculino     0.0           1.0   

  feridos_graves mortos      latitude     longitude regional delegacia  \
0            0.0    0.0  -20,24173903  -42,15868042  S